# GDCH Architecture, Data, Training, and Inference

This notebook is a self-contained walkthrough that builds the full GDCH stack end-to-end:

1. Define configuration (architecture + solver + training).
2. Process raw data into events/metadata.
3. Initialize model and run training.
4. Run inference utilities for next-event diagnostics.

It is independent of other notebooks and can be executed as-is when pointed at your dataset paths.

In [ ]:
import json
from pathlib import Path

import torch

from gdch import data as data_utils
from gdch import eval as eval_utils
from gdch import simulate as sim_utils
from gdch.train import train_from_config

print('Torch:', torch.__version__)

## 1) Configuration

Fill in the data paths for your dataset below. The architecture section shows the new configurable MLP stacks (e.g., `mlp_layers` and `time_mlp_layers`).

In [ ]:
config = {
    'data': {
        # Raw input CSV path for processing.
        'raw_input_path': 'data/cleaned_data.csv',
        'events_path': 'data/events.csv',
        'metadata_path': 'data/metadata.json',
        'distance_matrix_path': 'data/distance.npy',
        'train_split': 0.7,
        'val_split': 0.15,
    },
    'graph': {
        'sigma': 1400.0,
        'sigma_jump': 200.0,
        'knn': None,
    },
    'model': {
        'latent_dim': 32,
        'time_embed_dim': 32,
        'node_embed_dim': 8,
        'mlp_hidden_dim': 64,
        'time_hidden_dim': 64,
        # New architecture knobs.
        'mlp_layers': [128, 64],
        'time_mlp_layers': [64, 32],
        'mlp_dropout': 0.1,
        'mlp_layer_norm': True,
        'time_mlp_dropout': 0.1,
        'time_mlp_layer_norm': True,
        'alpha_init': 0.05,
        'beta_init': 0.1,
        'jump_eta': 0.0,
        'jump_tanh': True,
        'jump_scale': 1.0,
        'gate_use_z': True,
        'intensity_use_z': True,
        'eps': 1e-8,
        'baseline_init': 0.0,
        'baseline_from_data': True,
        'per_node_w': True,
    },
    'solver': {
        'method': 'dopri5',
        'rtol': 1e-4,
        'atol': 1e-6,
        'max_num_steps': 1000,
        'use_adjoint': False,
        'min_dt': 1e-6,
        'min_step': 1e-6,
        'time_dtype': 'float32',
        'fallback_method': 'rk4',
    },
    'regularization': {
        'lambda_z': 1e-5,
        'lambda_jump': 1e-5,
        'lambda_beta': 1e-6,
    },
    'training': {
        'epochs': 10,
        'chunk_size': 256,
        'eval_chunk_size': 1024,
        'lr': 3e-4,
        'weight_decay': 1e-3,
        'grad_clip': 2.0,
        'log_every': 20,
        'seed': 42,
        'device': 'cuda_if_available',
        'artifacts_dir': 'artifacts',
        'run_name': 'gdch_notebook',
    },
}

config


## 2) Data Processing

Convert the raw CSV into the `events.csv` + `metadata.json` format expected by GDCH.

In [ ]:
data_cfg = config['data']

metadata = data_utils.process_raw_csv(
    input_path=data_cfg['raw_input_path'],
    output_events_path=data_cfg['events_path'],
    output_metadata_path=data_cfg['metadata_path'],
    output_opo_metadata_path='data/opo_metadata.csv',
    output_distance_path=data_cfg['distance_matrix_path'],
    min_time_delta=1e-6,
)

metadata

## 3) Training

Train the model from the in-memory config (no external config file needed).

In [ ]:
run_dir = train_from_config(config)
print('Run artifacts saved to:', run_dir)

## 4) Inference

Load the best checkpoint and compute evaluation metrics + next-event quantities.

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

checkpoint_path = Path(run_dir) / 'checkpoint_best.pt'
config_path = Path(run_dir) / 'config.json'

# Reload config as saved during training.
config_reloaded = json.loads(config_path.read_text())

times, nodes = data_utils.load_processed_events(config_reloaded['data']['events_path'])
metadata = data_utils.load_metadata(config_reloaded['data']['metadata_path'])

# Build graph + model the same way eval does.
laplacian, jump_kernel = sim_utils._build_graph(
    config_reloaded['data'].get('distance_matrix_path', ''),
    config_reloaded.get('graph', {}),
    torch.device(device),
)
model = eval_utils._build_model(
    config_reloaded,
    metadata,
    torch.device(device),
    laplacian,
    jump_kernel,
)
checkpoint = torch.load(checkpoint_path, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

# Evaluate NLL on the full dataset as a simple sanity check.
val_nll = eval_utils.evaluate_nll(
    model,
    times.to(device),
    nodes.to(device),
    solver_config=config_reloaded.get('solver', {}),
    chunk_size=config_reloaded.get('training', {}).get('eval_chunk_size', 1024),
    warmup=None,
)
print('Full-data NLL per event:', val_nll)

# Example next-event location probabilities for the first event time.
with torch.no_grad():
    z0 = model.Z0
    lam, lam_sum = model.intensity(times[0].to(device), z0)
    probs = lam / lam_sum
print('Next-event location probs (first 5 nodes):', probs[:5].cpu().numpy())